In [ ]:
import os
# Must be set before importing ray and labtech.runners.ray
os.environ['RAY_DEDUP_LOGS'] = '0'

In [ ]:
from time import sleep

import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster
from pyspark.sql import SparkSession

import labtech
from labtech.storage import LocalStorage
from labtech.runners.ray import RayRunnerBackend

In [ ]:
# Ensure you first run: make spark-cluster
spark = (
    SparkSession.builder
    # Start a full Spark Classic connection (i.e. not using Spark Connect)
    # to provide a full SparkContext required for the ray cluster.
    .config('spark.cores.max', '3')
    .master('spark://localhost:7077')
    .appName('labtech_ray_demo')
    .getOrCreate()
)

In [ ]:
# Run a ray cluster on top of Spark.
setup_ray_cluster(
  max_worker_nodes=2,
  num_cpus_worker_node=1,
  num_gpus_worker_node=0,
  memory_worker_node=(512 * 1024 * 1024),
)
ray.init()

In [ ]:
@labtech.task
class Experiment:
    seed: int
    multiplier: int

    def run(self):
        labtech.logger.info(f'Running with seed {self.seed} and multiplier {self.multiplier}')
        sleep(1)
        return self.seed * self.multiplier


experiments = [
    Experiment(
        seed=seed,
        multiplier=multiplier,
    )
    for seed in range(10)
    for multiplier in range(4)
]

lab = labtech.Lab(
    storage=LocalStorage(
        'storage/ray_on_spark_lab',
        # The storage/ directory is mounted
        # at /opt/spark/shared-storage on
        # the Spark worker nodes:
        runner_dir='/opt/spark/shared-storage/ray_on_spark_lab',
    ),
    runner_backend=RayRunnerBackend(),
)

cached_experiments = lab.cached_tasks([Experiment])
print(f'{len(cached_experiments)} cached experiments.')

results = lab.run_tasks(experiments, bust_cache=True)
print(results)

In [ ]:
shutdown_ray_cluster()
ray.shutdown()